## 6. N-BEATS Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
    !pip install -q pytorch_forecasting
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 86.2 MB/s eta 0:00:00
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json

import numpy as np

from src.dataloader import load_cgm_stats, load_tensors, load_participant_ids
from src import metrics
from src import train
from src.models.nn_baseline import NBEATSPredictor, HORIZONS

MODEL_NAME = 'NBEATS'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'nbeats'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'nbeats_stability_summary.json'
VAL_STABILITY_SUMMARY_PATH = RESULTS_DIR / 'nbeats_val_stability_summary.json'

In [3]:
stats = load_cgm_stats()
train_loader = load_tensors('train')  # batch_size=256 default (Training Procedure)
val_loader = load_tensors('val')
test_loader = load_tensors('test')

val_participant_ids = load_participant_ids('val')
test_participant_ids = load_participant_ids('test')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print(f'number of train batches: ', len(train_loader), '\nnumber of valuation batches: ', len(val_loader),
      '\nnumber of test batches: ', len(test_loader))
print('number of test participants:', len(VAL_PARTICIPANT_IDS))
print('Corresponding participants ID: \n', VAL_PARTICIPANT_IDS)
print('number of test participants:', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

sample_cgm, sample_target = next(iter(train_loader))
print('\ninput batch shape (cgm):', tuple(sample_cgm.shape))
print('target batch shape:', tuple(sample_target.shape))

number of train batches:  327 
number of valuation batches:  66 
number of test batches:  66
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

input batch shape (cgm): (256, 24)
target batch shape: (256, 5)


In [5]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []  # one entry per seed, in memory -- also written to disk individually below

for seed in SEEDS:
    print(f"Validation results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = NBEATSPredictor(context_length=sample_cgm.shape[1])

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'nbeats' /f'nbeats_seed{seed}',
        verbose=False,
        progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader)
    y_pred_test, y_true_test = train.predict(model, test_loader)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])

    val_results_per_participant = metrics.evaluate_per_participant(
        y_pred_val, y_true_val, val_participant_ids, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'val_results_per_participant': val_results_per_participant,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'nbeats_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = val_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Validation results — NBEATS | seed=7

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: NBEATSPredictor  params: 75,312


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.13 | MAE 7.54 | gRMSE 12.01 | Zone A 96.55% | A+B 99.67%
 30 min | RMSE 17.64 | MAE 11.80 | gRMSE 19.80 | Zone A 89.66% | A+B 99.34%
 60 min | RMSE 25.06 | MAE 16.82 | gRMSE 29.54 | Zone A 81.39% | A+B 98.70%
 90 min | RMSE 29.40 | MAE 20.04 | gRMSE 35.17 | Zone A 75.95% | A+B 98.23%
120 min | RMSE 31.95 | MAE 22.10 | gRMSE 38.66 | Zone A 72.26% | A+B 97.93%

--------------------------------------------------------------------------------
Validation results — NBEATS | seed=14

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: NBEATSPredictor  params: 75,312


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.08 | MAE 7.51 | gRMSE 12.07 | Zone A 96.71% | A+B 99.67%
 30 min | RMSE 17.53 | MAE 11.59 | gRMSE 19.83 | Zone A 89.91% | A+B 99.31%
 60 min | RMSE 25.03 | MAE 16.45 | gRMSE 29.75 | Zone A 81.97% | A+B 98.66%
 90 min | RMSE 29.35 | MAE 19.49 | gRMSE 35.65 | Zone A 77.21% | A+B 98.17%
120 min | RMSE 31.92 | MAE 21.81 | gRMSE 39.03 | Zone A 72.78% | A+B 97.83%

--------------------------------------------------------------------------------
Validation results — NBEATS | seed=21

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: NBEATSPredictor  params: 75,312


...

 15 min | RMSE 11.18 | MAE 7.54 | gRMSE 12.31 | Zone A 96.76% | A+B 99.67%
 30 min | RMSE 17.66 | MAE 11.69 | gRMSE 20.15 | Zone A 90.00% | A+B 99.32%
 60 min | RMSE 25.09 | MAE 16.59 | gRMSE 30.00 | Zone A 81.87% | A+B 98.68%
 90 min | RMSE 29.34 | MAE 19.65 | gRMSE 35.68 | Zone A 76.84% | A+B 98.21%
120 min | RMSE 31.90 | MAE 21.82 | gRMSE 39.08 | Zone A 73.10% | A+B 97.82%

--------------------------------------------------------------------------------
Validation results — NBEATS | seed=63

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: NBEATSPredictor  params: 75,312


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.12 | MAE 7.53 | gRMSE 12.00 | Zone A 96.59% | A+B 99.67%
 30 min | RMSE 17.56 | MAE 11.60 | gRMSE 19.66 | Zone A 89.98% | A+B 99.34%
 60 min | RMSE 24.98 | MAE 16.33 | gRMSE 29.46 | Zone A 82.18% | A+B 98.66%
 90 min | RMSE 29.27 | MAE 19.48 | gRMSE 35.19 | Zone A 77.04% | A+B 98.18%
120 min | RMSE 31.88 | MAE 21.64 | gRMSE 38.65 | Zone A 73.40% | A+B 97.90%

--------------------------------------------------------------------------------
Validation results — NBEATS | seed=70

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: NBEATSPredictor  params: 75,312


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.19 | MAE 7.66 | gRMSE 12.32 | Zone A 96.52% | A+B 99.62%
 30 min | RMSE 17.62 | MAE 11.91 | gRMSE 19.90 | Zone A 89.56% | A+B 99.32%
 60 min | RMSE 25.05 | MAE 17.03 | gRMSE 29.55 | Zone A 81.04% | A+B 98.70%
 90 min | RMSE 29.30 | MAE 20.00 | gRMSE 35.29 | Zone A 76.07% | A+B 98.17%
120 min | RMSE 31.89 | MAE 21.98 | gRMSE 38.87 | Zone A 72.66% | A+B 97.86%

--------------------------------------------------------------------------------


In [6]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

def build_summary(results_key, participant_key, participant_ids):
    """results_key/participant_key: ('val_results', 'val_results_per_participant') or
    ('test_results', 'test_results_per_participant')."""
    summary = {}
    for h in HORIZONS:
        summary[h] = {}
        for m in METRICS:
            if m == 'zone_a':
                seed_values = [r[results_key][h]['ceg']['A'] for r in all_results]
            elif m == 'zone_ab':
                seed_values = [r[results_key][h]['ceg']['A'] + r[results_key][h]['ceg']['B']
                                for r in all_results]
            else:
                seed_values = [r[results_key][h][m] for r in all_results]

            mean = float(np.mean(seed_values))
            sd1 = float(np.std(seed_values, ddof=1))

            participant_means = []
            for pid in participant_ids:
                values_across_seeds = [_extract(r[participant_key][pid][h], m)
                                        for r in all_results]
                participant_means.append(np.mean(values_across_seeds))
            sd2 = float(np.std(participant_means, ddof=1))

            summary[h][m] = {
                'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2,
            }
    return summary

In [7]:
val_summary = build_summary('val_results', 'val_results_per_participant', VAL_PARTICIPANT_IDS)
test_summary = build_summary('test_results', 'test_results_per_participant', PARTICIPANT_IDS)

In [8]:
def _print_summary(split_label, summary):
    print(f'=== {MODEL_NAME} {split_label} set: overall mean +/- SD_seed (SD_participant) ===')
    for h in HORIZONS:
        s = summary[h]
        print(
            f"{h:>3}-min:  "
            f"RMSE = {s['rmse']['mean']:.2f} +/- "
            f"{s['rmse']['sd1_across_seeds']:.2f} "
            f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
            f"MAE = {s['mae']['mean']:.2f} +/- "
            f"{s['mae']['sd1_across_seeds']:.2f} "
            f"({s['mae']['sd2_across_participants']:.2f})   "
            f"gRMSE = {s['grmse']['mean']:.2f} +/- "
            f"{s['grmse']['sd1_across_seeds']:.2f} "
            f"({s['grmse']['sd2_across_participants']:.2f})   "
            f"Zone A = {s['zone_a']['mean']:.2f} +/- "
            f"{s['zone_a']['sd1_across_seeds']:.2f} "
            f"({s['zone_a']['sd2_across_participants']:.2f})%   "
            f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
            f"{s['zone_ab']['sd1_across_seeds']:.2f} "
            f"({s['zone_ab']['sd2_across_participants']:.2f})%"
        )

_print_summary('validation', val_summary)

=== NBEATS validation set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 11.15 +/- 0.06 (2.36) mg/dL   MAE = 7.58 +/- 0.07 (1.60)   gRMSE = 12.19 +/- 0.14 (3.28)   Zone A = 96.59 +/- 0.14 (3.33)%   Zone A+B = 99.64 +/- 0.03 (0.71)%
 30-min:  RMSE = 17.60 +/- 0.05 (3.83) mg/dL   MAE = 11.73 +/- 0.12 (2.45)   gRMSE = 19.91 +/- 0.15 (5.78)   Zone A = 89.81 +/- 0.19 (5.43)%   Zone A+B = 99.32 +/- 0.03 (0.98)%
 60-min:  RMSE = 25.04 +/- 0.04 (7.42) mg/dL   MAE = 16.65 +/- 0.22 (4.72)   gRMSE = 29.64 +/- 0.18 (11.16)   Zone A = 81.61 +/- 0.40 (5.11)%   Zone A+B = 98.68 +/- 0.02 (1.45)%
 90-min:  RMSE = 29.32 +/- 0.04 (10.34) mg/dL   MAE = 19.75 +/- 0.22 (6.89)   gRMSE = 35.35 +/- 0.19 (15.26)   Zone A = 76.55 +/- 0.45 (7.27)%   Zone A+B = 98.20 +/- 0.03 (1.97)%
120-min:  RMSE = 31.90 +/- 0.04 (12.17) mg/dL   MAE = 21.88 +/- 0.19 (8.34)   gRMSE = 38.81 +/- 0.16 (18.00)   Zone A = 72.88 +/- 0.47 (8.37)%   Zone A+B = 97.88 +/- 0.04 (2.49)%


In [9]:
with open(VAL_STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': VAL_PARTICIPANT_IDS,
        'per_horizon_summary': val_summary,
    }, f, indent=2)

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': test_summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/lstm_seed*_results.json')
print(f'saved validation (selection) summary to {VAL_STABILITY_SUMMARY_PATH}')
print(f'saved test (confirmatory-only) summary to {STABILITY_SUMMARY_PATH}')


saved per-seed results to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/nbeats/lstm_seed*_results.json
saved validation (selection) summary to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/nbeats/nbeats_val_stability_summary.json
saved test (confirmatory-only) summary to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/nbeats/nbeats_stability_summary.json
